In [22]:
import torch
import pandas as pd

In [27]:
df = pd.read_csv("../oritdb_pairs.csv")

In [28]:
df

,oriTDB_accession,oriTDB_id,nic site,relaxase_family,relaxase_sequence,host
0,100001,GAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTA...,60..61,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Pseudomonas aeruginosa
1,100002,CCGGCCAGCCTCGCAGAGCAGGATTCCCGTTGAGCACCGCCAGGTG...,107..108,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Klebsiella aerogenes K8841
2,100003,CCAGTTTCTCGAAGAGAAACCGGTAAGTGCGCCCTCCC,31..32,MobA,MAIYHLTAKTGSRSGGQSARAKADYIQREGKYARDMDEVLHAESGH...,Escherichia coli
3,100004,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli
4,100005,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli
...,...,...,...,...,...,...
4400,104401,CTCACCACCAAAAGCTCCACACCCCACGCAAAATTAAAGTTTTGCT...,_,traI_HV236_RS22985,MMSIGSVKSAGSAGNYYTDKDNYYVIGSMGERWAGKGAEALGLSGG...,Enterobacter hormaechei strain RHBSTW-00564
4401,104402,GGGTTTCGGGGCGCAGCCCTGAACCAGTCACGTAGCGCTAGCGGAG...,_,NaN,NaN,Enterobacter hormaechei strain RHBSTW-00564
4402,104403,AAATCTGCAAAATTTTAATTTTGCGTAGTGTGTGGTGATTTTGTGG...,33..34,NaN,NaN,Klebsiella pneumoniae strain RHBSTW-00574
4403,104404,AAATCCGCAAAATTTTAATTTTGCGTAGTGTGTGGATATTTTTAGT...,_,NaN,NaN,Klebsiella pneumoniae strain RHBSTW-00574


In [29]:

filtered_df = df.copy()

filtered_df = filtered_df[filtered_df["relaxase_sequence"].notna() & (df["nic site"] != '_')]
filtered_df = filtered_df[filtered_df["oriTDB_id"].notna()]

filtered_df = filtered_df[filtered_df["relaxase_sequence"].str.len() > 100]


In [30]:
print(len(filtered_df))
filtered_df

1403


,oriTDB_accession,oriTDB_id,nic site,relaxase_family,relaxase_sequence,host
0,100001,GAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTA...,60..61,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Pseudomonas aeruginosa
1,100002,CCGGCCAGCCTCGCAGAGCAGGATTCCCGTTGAGCACCGCCAGGTG...,107..108,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Klebsiella aerogenes K8841
2,100003,CCAGTTTCTCGAAGAGAAACCGGTAAGTGCGCCCTCCC,31..32,MobA,MAIYHLTAKTGSRSGGQSARAKADYIQREGKYARDMDEVLHAESGH...,Escherichia coli
3,100004,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli
4,100005,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli
...,...,...,...,...,...,...
4348,104349,TATTGAATGTTTTATTCTTTAAATCAGTTAGTTAGTGCGATTTATC...,62..63,MobF,MLDITTITRIDIASVVGYYADAKDDYYSKDSSFTAWHGQGAEALGL...,Enterobacter kobei strain 2020CK-00216
4355,104356,GGATTTAGGTTTTTTTTTAATCGCTTCACATTTCGTTAGCATGCGA...,77..78,MobH,MMMNFRALYLCIKRILGIFSSQENDATSVMIEDISSLSPFAQILGD...,Escherichia coli strain NB4833
4367,104368,TCAGCTGAAACTTTAAATATGAGTGTGCCTACTTTCGTTAAGAAAA...,66..67,Rep_trans_PUW68_RS12235_pShVSI35,MSTENYSNYLQNKDLDNFSKTGYSNSRLDAHTVCISKPKISFDAMT...,Staphylococcus haemolyticus strain VSI35
4371,104372,AATTACATGATTTAAAACGTAAATCAGCAAAAACTTGTTTTTGCGT...,53..54,traI_LVJ26_RS24385,MLSFSVVKSAGSAGNYYTDKDNYYVLGSMGERWAGQGAEQLGLQGS...,Escherichia coli strain BY68


In [11]:
import torch
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

ESMC: Transformer Engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically - measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install an xformers build matching your PyTorch and CUDA versions for a fused attention kernel.
ESMC: flash-attn rotary kernel not installed; falling back to pure-PyTorch RoPE. For faster GPU inference run `pip install flash-attn

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [12]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings

In [13]:
nes_seq = """
MAMYHFQNKFVSKANGQSATAKSAYNSASRIKDFKENEFKDYSNKQCDYSEILLPNNADD
KFKDREYLWNKVHDVENRKNSQVAREIIIGLPNEFDPNSNIELAKEFAESLSNEGMIVDL
NIHKINEENPHAHLLCTLRGLDKNNEFEPKRKGNDYIRDWNTKEKHNEWRKRWENVQNKH
LEKNGFSVRVSADSYKNQNIDLEPTKKEGWKARKFEDETGKKSSISKHNESIKKRNQQKI
DKMFNEVDDMKSHKLNAFSYMNKSDSTTLKNMAKDLKIYVTPINMYKENERLYDLKQKTS
LITDDEDRLNKIEDIEDRQKKLESINEVFEKQAGIFFDKNYPDQSLNYSDDEKSLSHMHY
INDRDVLPANNELEDIVKEKRIKEAQISLNTVLGNRDISLESIEKESNFFADKLSNILEK
NNLSFDDVLENKHEGMEDSLKIDYYTNKLEVFRNAENILEDYYDVQIKELFTDDEDYKAF
NEVTDIKEKQQLIDFKTYHGTENTIEMLETGNFIPKYSDEDRKYITEQVKLLQEKEFKPN
KNQHDKFVFGAIQKKLLSEYDFDYSDNNDLKHLYQESNEVGDEISKDNIEEFYEGNDILV
DKETYNNYNRKQQAYGLINAGLDSMIFNFNEIFRERMPKYINHQYKGKNHSKQRHELKNK
RGMHL
"""

nes_embed = get_protein_embedding(nes_seq)

nes_sDBM = nes_embed[4:13, :]

In [ ]:

# nobaseL -> no of bases before nic site, nobaseR -> no of bases after nic site
def get_nic_seq(row, nobaseL = 7, nobaseR = 2):
    nic_site = row["nic site"]

    # assumes consecutive pos e.g. 4..5 -> always true for dataset
    nic_pos = int(nic_site.split(".")[0])

    oriT_seq = row["oriTDB_id"]

    # account for the fact that indexing in nic_pos begins at 1 (not 0)
    nic_site_start = nic_pos - nobaseL
    nic_site_end = nic_pos + nobaseR # excluded 

    if nic_pos < nobaseL or len(oriT_seq) <= (nic_pos + nobaseR - 1):
        return "NaN"

    return oriT_seq[nic_site_start: nic_site_end]



def query_sDBM(row, known_sDBM = nes_sDBM):

    protein_embed = get_protein_embedding(row["relaxase_sequence"])

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    next_start = 0
    next_end = 0 

    for i in range(0, 30):
        match_block = protein_embed[i:i+9, :]

        sim = F.cosine_similarity(match_block, known_sDBM, dim = 1)

        mean = torch.mean(sim)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + 9
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean
            next_start = i + 1
            next_end = i + 9

    perc = (max_mean - next_max_mean) * 100

    sDBM = row["relaxase_sequence"][start_seq : end_seq + 1]

    return sDBM, (start_seq, end_seq), float(perc)

    
filtered_df[["sDBM", "sDBM_pos", "sDBM_sim_gap"]] = (
    filtered_df.apply(query_sDBM, axis=1, result_type="expand")
)

filtered_df['oriT_nic_seq'] = filtered_df.apply(get_nic_seq, axis = 1)

filtered_df


# 2m 46 s



,oriTDB_accession,oriTDB_id,nic site,relaxase_family,relaxase_sequence,host,sDBM,sDBM_pos,sDBM_sim_gap,oriT_nic_seq
0,100001,GAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTA...,60..61,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Pseudomonas aeruginosa,HVPMRSIKK,"(4, 12)",3.985026,TATCCTGCC
1,100002,CCGGCCAGCCTCGCAGAGCAGGATTCCCGTTGAGCACCGCCAGGTG...,107..108,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Klebsiella aerogenes K8841,HVPMRSIKK,"(4, 12)",3.985026,TATCCTGCC
2,100003,CCAGTTTCTCGAAGAGAAACCGGTAAGTGCGCCCTCCC,31..32,MobA,MAIYHLTAKTGSRSGGQSARAKADYIQREGKYARDMDEVLHAESGH...,Escherichia coli,HLTAKTGSR,"(4, 12)",16.299099,AAGTGCGCC
3,100004,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli,VEKDYRVKR,"(4, 12)",0.642514,TATCCTGCA
4,100005,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli,VEKDYRVKR,"(4, 12)",0.642514,TATCCTGCA
...,...,...,...,...,...,...,...,...,...,...
4306,104307,GGATTTAGGTTTTTTTTTAATCGCTTCACATTTCGTTAGCATGCGC...,78..79,MobH,MMMNFRALYLCIKRILGIFSSQENDATSVMIEDISSLSPFAQILGD...,Enterobacter kobei strain 12379-yvys,NFRALYLCI,"(3, 11)",0.067639,AGTTTGGTT
4308,104309,GGATTTAGGTTTTTTTTTAATCGCTTCACATTTCGTTAGCATGCGC...,78..79,MobH,MMMNFRALYLCIKRILGIFSSQENDATSVMIEDISSLSPFAQILGD...,Enterobacter asburiae strain 16773-yvys,NFRALYLCI,"(3, 11)",0.067639,AGTTTGGTT
4311,104312,GGATTTAGGTTTTTTTTTAATCGCTTCACATTTCGTTAGCATGCGC...,78..79,MobH,MMMNFRALYLCIKRILGIFSSQENDATSVMIEDISSLSPFAQILGD...,Enterobacter kobei strain 11743-yvys,NFRALYLCI,"(3, 11)",0.067639,AGTTTGGTT
4312,104313,TATTTCATTTTTTGTTTTTTTAATCAGTTAGTTAGTGCGATTCATC...,62..63,MobF,MLDITTITRIDIASVVGYYADAKDDYYSKDSSFTAWHGQGAEALGL...,Enterobacter kobei strain 11743-yvys,SKDSSFTAW,"(27, 35)",0.767967,TAGGTGTAT


In [21]:
filtered_df.to_csv('../data/processed/relaxase_oriT_dataset.csv', index=False)

In [11]:
filtered_df['relaxase_length'] = filtered_df['relaxase_sequence'].str.len()

filtered_df

,oriTDB_accession,oriTDB_id,nic site,relaxase_family,relaxase_sequence,host,sDBM,sDBM_pos,sDBM_sim_gap,oriT_nic_seq,relaxase_length
0,100001,GAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTA...,60..61,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Pseudomonas aeruginosa,HVPMRSIKK,"(4, 12)",<built-in method item of Tensor object at 0x7d...,TATCCTGCC,732
1,100002,CCGGCCAGCCTCGCAGAGCAGGATTCCCGTTGAGCACCGCCAGGTG...,107..108,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Klebsiella aerogenes K8841,HVPMRSIKK,"(4, 12)",<built-in method item of Tensor object at 0x7d...,TATCCTGCC,732
2,100003,CCAGTTTCTCGAAGAGAAACCGGTAAGTGCGCCCTCCC,31..32,MobA,MAIYHLTAKTGSRSGGQSARAKADYIQREGKYARDMDEVLHAESGH...,Escherichia coli,HLTAKTGSR,"(4, 12)",<built-in method item of Tensor object at 0x7d...,AAGTGCGCC,700
3,100004,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli,VEKDYRVKR,"(4, 12)",<built-in method item of Tensor object at 0x7d...,TATCCTGCA,385
4,100005,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli,VEKDYRVKR,"(4, 12)",<built-in method item of Tensor object at 0x7d...,TATCCTGCA,385
...,...,...,...,...,...,...,...,...,...,...,...
3831,103832,GAATAAGGGACAGTGAAGATAGATAACCGGCTCGCCGGTTAGCTAA...,60..61,Relaxase_L3V85_RS37255_p_IPSP_MA,MIAKHVPMRSLGKSDFAGLANYITDAQSKDHRLGHVQATNCEAGSI...,Variovorax paradoxus strain IPSP_MA_2,HVPMRSLGK,"(4, 12)",<built-in method item of Tensor object at 0x7d...,CATCCTGCC,746
3834,103835,TTCTTTGTTTCCCTGTCAAATACATTGTGGTTTTGATAAAATCATC...,315..316,MobP1,MGVYVDKEFRVKRKSSEAGRKSAFAHKVKNGGKNYQRNVQERINRK...,Salmonella enterica subsp. enterica serovar Ty...,VDKEFRVKR,"(4, 12)",<built-in method item of Tensor object at 0x7d...,TATCCTGCA,386
3867,103868,CTTTGTTTACCTGTTAAGTACATCGTTGTTTTGATAAAATCATTAT...,313..314,MobP1,MGVYVDKEYRVKRKSSENGRKSAFAHKVKNGGKNYSRNVQERINRK...,Salmonella enterica subsp. enterica serovar En...,VDKEYRVKR,"(4, 12)",<built-in method item of Tensor object at 0x7d...,TATCCTGCA,388
3868,103869,GGTTATTGCTACTTAATGCCGATAACGACTCAGGCTTTGAGGTTTT...,109..110,MobH,MNFRALFLSMQRVFGIFSRRENDVSELMMKDAANFSPFAQIIGEQK...,Salmonella enterica subsp. enterica serovar Ty...,NFRALFLSM,"(1, 9)",<built-in method item of Tensor object at 0x7d...,AGTTTGGTT,1011


In [41]:
from IPython.display import display, HTML

# Wrap df.to_html() in a scrollable div using IPython display
with pd.option_context('display.max_rows', None):
    display(HTML("<div style='height: 300px; width: 100%; overflow: auto;'>" + filtered_df.to_html() + "</div>"))

,oriTDB_accession,oriTDB_id,nic site,relaxase_family,relaxase_sequence,host
0,100001,GAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTACTTCACCTATCCTGCCCGGCTGACGCCGTTGGATACACCAAGGAAAGTCTACA,60..61,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTLPAVMAEVMATQHGNTRSEADKTYHLLVSFRAGEKPDAETLRAIEDRICAGLGFAEHQRVSAVHHDTDNLHIHIAINKIHPTRNTIHEPYRAYRALADLCATLERDYGLERDNHETRQRVSENRANDMERHAGVESLVGWIKRECLPELQAAQSWEDLHRVLRENGLKLRERGNGFIFEAGDGTTVKASTVSRDLSKPKLEARFGAFTPAEGGEAPRRREYRAKPLKTRIDTTELYARYQSERQEMGAVRKGELDTLRRRRDRLIEAAMRSNRLRRAAIKLLGEGRIAKRLMYAQAHKALRADLDKINREYRQGRQAVQERTQRRAWADWLKAEAMKGDDKALAALRAREGRSDLKGNTIQGSGEAKPGHAAVTDNITKKGTIIYRVGSSAVRDDGDRLQVSREATTDGLDAALRLAMERFGDRITVNGTAEFKERIAQAAAAGRLAITFDDAALERRRQELLTKEQAHEQPERNDGRRDRGGDGGIRPAAARTTLNATGGDGDRRDARAVSAGGTVALRKPNVGRIGRKPPPQSQNRLRALSQLGVVRIAGGAEMLLPRDVPGHVEQQGAEPAHALRRGVSGPGRGLKPEQIAAAEKYVAEREQKRLNGFDIPKHARYTDYVGALSYAGTRNVEDQALALLRKENDEILVLPVDKATVQRMKRLAIGDPVTVTPRGSLKTTRGRSR,Pseudomonas aeruginosa
1,100002,CCGGCCAGCCTCGCAGAGCAGGATTCCCGTTGAGCACCGCCAGGTGCGAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTACTTCACCTATCCTGCCCGG,107..108,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTLPAVMAEVMATQHGNTRSEADKTYHLLVSFRAGEKPDAETLRAIEDRICAGLGFAEHQRVSAVHHDTDNLHIHIAINKIHPTRNTIHEPYRAYRALADLCATLERDYGLERDNHETRQRVSENRANDMERHAGVESLVGWIKRECLPELQAAQSWEDLHRVLRENGLKLRERGNGFIFEAGDGTTVKASTVSRDLSKPKLEARFGAFTPAEGGEAPRRREYRAKPLKTRIDTTELYARYQSERQEMGAVRKGELDTLRRRRDRLIEAAMRSNRLRRAAIKLLGEGRIAKRLMYAQAHKALRADLDKINREYRQGRQAVQERTQRRAWADWLKAEAMKGDDKALAALRAREGRSDLKGNTIQGSGEAKPGHAAVTDNITKKGTIIYRVGSSAVRDDGDRLQVSREATTDGLDAALRLAMERFGDRITVNGTAEFKERIAQAAAAGRLAITFDDAALERRRQELLTKEQAHEQPERNDGRRDRGGDGGIRPAAARTTLNATGGDGDRRDARAVSAGGTVALRKPNVGRIGRKPPPQSQNRLRALSQLGVVRIAGGAEMLLPRDVPGHVEQQGAEPAHALRRGVSGPGRGLKPEQIAAAEKYVAEREQKRLNGFDIPKHARYTDYVGALSYAGTRNVEDQALALLRKENDEILVLPVDKATVQRMKRLAIGDPVTVTPRGSLKTTRGRSR,Klebsiella aerogenes K8841
2,100003,CCAGTTTCTCGAAGAGAAACCGGTAAGTGCGCCCTCCC,31..32,MobA,MAIYHLTAKTGSRSGGQSARAKADYIQREGKYARDMDEVLHAESGHMPEFVERPADYWDAADLYERANGRLFKEVEFALPVELTLDQQKALASEFAQHLTGAERLPYTLAIHAGGGENPHCHLMISERINDGIERPAAQWFKRYNGKTPEKGGAQKTEALKPKAWLEQTREAWADHANRALERAGHDARIDHRTLEAQGIERLPGVHLGPNVVEMEGRGIRTDRADVALNIDTANAQIIDLQEYREAIDHERNRQSEEIQRHQRVSGADRTAGPEHGDTGRRSPAGHEPDPAGQRGAGGGVAESPAPDRGGMGGAGQRVAGGSRRGEQRRAERPERVAGVALEAMANRDAGFHDAYGGAADRIVALARPDATDNRGRLDLAALGGPMKNDRTLQAIGRQLKAMGCERFDIGVRDATTGQMMNREWSAAEVLQNTPWLKRMNAQGNDVYIRPAEQERHGLVLVDDLSEFDLDDMKAEGREPALVVETSPKNYQAWVKVADAAGGELRGQIARTLASEYDADPASADSRHYGRLAGFTNRKDKHTTRAGYQPWVLLRESKGKTATAGPALVQQAGQQIEQAQRQQEKARRLASLELPERQLSRHRRTALDEYRSEMAGLVKRFGDDLSKCDFIAAQKLASRGRSAEEIGKAMAEASPALAERKPGHEADYIERTVSKVMGLPSVQLARAELARAPAPRQRGM,Escherichia coli
3,100004,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRKGASKEVVVKISGGAVTRQGIKNSIDYMSRESQLSVMNESGQVWQGEEIKEAKEHMVDAANDAQHVHDDKGRENKKITQNIVFSPPVSAKVKPEGLLDSVRKTLNKKYPNNRFVLGYHEDKKEHPHVHVVFRIKDTDGKRTDIKKKDLREIRTGFCNELILKGYDVKATHKQTQGLNQSIKDRHASAPKRQKGVYEVVDVGYDHYQHDKSNPKQHFLKLKTLNKGVEKTYWGAGFGDLVQRESVKKGDLVKLKKLGQKEVKIPALDKNGIQQGWKTAHRNDWQLQNLGVKGIDRKLSNDKSFKLDDISSIQKQQQQMKKFNLVKQNMLQSQQKIKVGIKF,E. coli
4,100005,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRKGASKEVVVKISGGAVTRQGIKNSIDYMSRESQLSVMNESGQVWQGEEIKEAKEHMVDAANDAQHVHDDKGRENKKITQNIVFSPPVSAKVKPEGLLDSVRKTLNKKYPNNRFVLGYHEDKKEHPHVHVVFRIKDTDGKRTDIKKKDLREIRTGFCNELILKGYDVKATHKQTQGLNQSIKDRHASAPKRQKGVYEVVDVGYDHYQHDKSNPKQHFLKLKTLNKGVEKTYWGAGFGDLVQRESVKKGDLVKLKKLGQKEVKIPALDKNGIQQGWKTAHRNDWQLQNLGVKGIDRKLSNDKSFKLDDISSIQKQQQQMKKFNLVKQNMLQSQQKIKVGIKF,E. coli
5,100006,ATACGAAGTAACGAAGTTACTGCGTATAAGTGCGCCCT,34..35,TraA,MTIAKRENGKRSLIAMASYRSGEKLYSELYEKTNLYNHRTVKPEAFILKPDYVPNEFLDRQTLWNKMELAEKSPNAQLCREVNVALPIELNNSDQRMLIEDFVKDNFVNEGMIADVAIHRDDENNPHAHIMLTMREVDSEGNILNKSHRIPKLDENGNQIFNEKGQRVTVSIKTNDWGRKSLVSEIRKDWADKVNQYLKDRNIDQQITEKSHAELGKKELPTIHEGFYSKKLEDKGVISELKRKNLEIQSYNDILAELDKLENQEKVLKQDQNFTLKFEKTFSPLEKGELKNLSKELKLFINDENIDKRLGELKRWENSLIFNNKMEIQKQRLMLSKISSERDMLTKANEILDKQAERFFKKSYPSLNIDKFSNHEVRAMVNETIFRKQLLNKDQLAEVIYNERVVEKEESKKIFKEKPFQTSRYLDSKIKQIEDSITKENNP